In [1]:
pip install rasterio

Note: you may need to restart the kernel to use updated packages.


In [3]:
#The file we need is gis_osm_roads_free_1.shp — that's the road network. The rest is buildings, water, railways, etc. which we don't need.
#Let's load it and see what we're working with:

import geopandas as gpd

roads = gpd.read_file(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\ghana-260322-free.shp\gis_osm_roads_free_1.shp")

print(f"Shape: {roads.shape}")
print(f"\nColumns: {list(roads.columns)}")
print(f"\nRoad types:")
print(roads['fclass'].value_counts())

Shape: (373884, 11)

Columns: ['osm_id', 'code', 'fclass', 'name', 'ref', 'oneway', 'maxspeed', 'layer', 'bridge', 'tunnel', 'geometry']

Road types:
fclass
residential       258859
service            36724
unclassified       23243
path               19822
track              17257
tertiary            4931
footway             4863
secondary           2528
trunk               1897
primary             1689
trunk_link           482
primary_link         295
secondary_link       228
living_street        218
steps                172
track_grade4         112
track_grade3         110
tertiary_link         93
pedestrian            84
motorway_link         60
track_grade5          60
motorway              43
track_grade2          42
unknown               36
cycleway              17
bridleway             12
track_grade1           5
busway                 2
Name: count, dtype: int64


In [4]:
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio

pd.set_option('display.max_columns', None)

# 1. Load master dataset (facilities)
master = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv")
print(f"✅ Facilities: {master.shape[0]} facilities")

# 2. Load road network
roads = gpd.read_file(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\ghana-260322-free.shp\gis_osm_roads_free_1.shp")
print(f"✅ Roads: {roads.shape[0]} road segments")

# 3. Load population density grid
pop_raster = rasterio.open(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\gha_pd_2020_1km_UNadj.tif")
pop_data = pop_raster.read(1)
print(f"✅ Population grid: {pop_data.shape} ({pop_data.shape[0]} rows x {pop_data.shape[1]} columns)")
print(f"   Total population in grid: {int(pop_data[pop_data > 0].sum()):,}")

✅ Facilities: 9978 facilities
✅ Roads: 373884 road segments
✅ Population grid: (773, 533) (773 rows x 533 columns)
   Total population in grid: 36,950,968


In [5]:
# Road types in our data and what they mean:

# === MAJOR ROADS (for cars, trucks, buses) ===
# motorway        — Highway/expressway, like Accra-Tema motorway. Fastest roads, limited access
# motorway_link   — On/off ramps connecting to motorways
# trunk           — Major national highways, like the N1. Main roads connecting cities
# trunk_link      — On/off ramps connecting to trunk roads
# primary         — Main regional roads connecting large towns
# primary_link    — Connectors to primary roads
# secondary       — Roads connecting smaller towns to each other
# secondary_link  — Connectors to secondary roads
# tertiary        — Local roads connecting villages to towns
# tertiary_link   — Connectors to tertiary roads

# === URBAN/TOWN ROADS ===
# residential     — Roads within towns and neighborhoods where people live
# living_street   — Very slow residential streets, shared with pedestrians
# service         — Access roads to buildings, parking lots, gas stations
# busway          — Roads reserved for buses only

# === RURAL/UNPAVED ROADS ===
# unclassified    — Minor roads, often unpaved, connecting small communities
# track           — Unpaved rural roads, often used by farms and villages
# track_grade1    — Best condition unpaved track (compacted gravel)
# track_grade2    — Decent unpaved track
# track_grade3    — Rough unpaved track
# track_grade4    — Very rough, barely maintained
# track_grade5    — Worst condition, almost impassable by car

# === PEDESTRIAN/NON-VEHICLE ===
# path            — General walking path, could be through bush or fields
# footway         — Paved or defined walking path, like a sidewalk
# pedestrian      — Streets closed to cars, pedestrian only zones
# steps           — Staircases
# cycleway        — Bicycle lanes or paths
# bridleway       — Paths for horses

# === OTHER ===
# unknown         — Road type not classified

In [6]:
import pandas as pd
import geopandas as gpd

# Load roads and master
master = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv")

roads = gpd.read_file(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\ghana-260322-free.shp\gis_osm_roads_free_1.shp")

# Load district polygons
districts_gdf = gpd.read_file(r"C:\Users\hp\Downloads\Code & Scripts\gadm41_GHA_2.shp")
districts_gdf = districts_gdf.to_crs(roads.crs)

# Get the midpoint of each road and assign it to a district
roads['midpoint'] = roads.geometry.interpolate(0.5, normalized=True)
roads_points = roads.set_geometry('midpoint')

roads_with_district = gpd.sjoin(roads_points, districts_gdf[['NAME_1', 'NAME_2', 'geometry']], how='left', predicate='within')

print(f"Roads assigned to a district: {roads_with_district['NAME_2'].notna().sum():,}")
print(f"Roads not in any district: {roads_with_district['NAME_2'].isna().sum():,}")

C:\Users\hp\AppData\Local\Temp\ipykernel_24448\317097341.py:14: UserWarning: Geometry is in a geographic CRS. Results from 'interpolate' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  roads['midpoint'] = roads.geometry.interpolate(0.5, normalized=True)


Roads assigned to a district: 370,169
Roads not in any district: 3,715


In [7]:
# Categorize roads
def road_category(fclass):
    if fclass in ['motorway', 'motorway_link', 'trunk', 'trunk_link', 'primary', 'primary_link']:
        return 'Major Roads'
    elif fclass in ['secondary', 'secondary_link', 'tertiary', 'tertiary_link']:
        return 'Connecting Roads'
    elif fclass in ['residential', 'living_street', 'service', 'busway']:
        return 'Urban/Town Roads'
    elif fclass in ['unclassified', 'track', 'track_grade1', 'track_grade2', 'track_grade3', 'track_grade4', 'track_grade5', 'unknown']:
        return 'Rural/Unpaved'
    elif fclass in ['path', 'footway', 'pedestrian', 'steps', 'cycleway', 'bridleway']:
        return 'Walking/Non-vehicle'
    else:
        return 'Other'

roads_with_district['category'] = roads_with_district['fclass'].apply(road_category)

# National overview first
print("=== NATIONAL ROAD BREAKDOWN ===\n")
cat_counts = roads_with_district['category'].value_counts()
for cat, count in cat_counts.items():
    pct = round(count / len(roads_with_district) * 100, 1)
    print(f"  {cat:<25} {count:>8}   ({pct}%)")

# Now by region — what percentage of roads are walking-only?
print(f"\n\n=== WALKING/NON-VEHICLE ROADS BY REGION ===\n")
region_roads = roads_with_district.groupby('NAME_1')['category'].value_counts().unstack(fill_value=0)
region_roads['Total'] = region_roads.sum(axis=1)
region_roads['Walking_Pct'] = round(region_roads.get('Walking/Non-vehicle', 0) / region_roads['Total'] * 100, 1)
region_roads['Rural_Pct'] = round(region_roads.get('Rural/Unpaved', 0) / region_roads['Total'] * 100, 1)

print(region_roads[['Total', 'Walking/Non-vehicle', 'Walking_Pct', 'Rural/Unpaved', 'Rural_Pct']].sort_values('Walking_Pct', ascending=False).to_string())

=== NATIONAL ROAD BREAKDOWN ===

  Urban/Town Roads            295803   (79.1%)
  Rural/Unpaved                40865   (10.9%)
  Walking/Non-vehicle          24970   (6.7%)
  Connecting Roads              7780   (2.1%)
  Major Roads                   4466   (1.2%)


=== WALKING/NON-VEHICLE ROADS BY REGION ===

category        Total  Walking/Non-vehicle  Walking_Pct  Rural/Unpaved  Rural_Pct
NAME_1                                                                           
Oti              4581                 1617         35.3           1457       31.8
Upper East      11249                 3209         28.5           1631       14.5
Savannah         9385                 1967         21.0           3734       39.8
North East       2154                  433         20.1            473       22.0
Upper West      11851                 2051         17.3           2256       19.0
Northern        23105                 2870         12.4           5849       25.3
Ahafo            5882           

In [8]:
# District level — which districts have the highest percentage of walking-only roads?
district_roads = roads_with_district.groupby(['NAME_2', 'NAME_1'])['category'].value_counts().unstack(fill_value=0)
district_roads['Total'] = district_roads.sum(axis=1)
district_roads['Walking_Pct'] = round(district_roads.get('Walking/Non-vehicle', 0) / district_roads['Total'] * 100, 1)

print("=== TOP 20 DISTRICTS BY WALKING PATH DEPENDENCY ===\n")
top_walking = district_roads.sort_values('Walking_Pct', ascending=False).head(20)
print(f"{'District':<30} {'Region':<15} {'Total Roads':<15} {'Walking Paths':<15} {'Walking %'}")
print("="*90)
for (district, region), row in top_walking.iterrows():
    walking = int(row.get('Walking/Non-vehicle', 0))
    print(f"{district:<30} {region:<15} {int(row['Total']):<15} {walking:<15} {row['Walking_Pct']}%")

=== TOP 20 DISTRICTS BY WALKING PATH DEPENDENCY ===

District                       Region          Total Roads     Walking Paths   Walking %
Tempane                        Upper East      1649            1367            82.9%
Krachi Nchumuru                Oti             1489            1047            70.3%
Zabzugu                        Northern        1233            790             64.1%
East Gonja                     Savannah        479             302             63.0%
Krachi East                    Oti             659             312             47.3%
Garu                           Upper East      315             146             46.3%
Talensi                        Upper East      630             277             44.0%
Wa East                        Upper West      385             140             36.4%
West Gonja                     Savannah        2077            752             36.2%
Kpandai                        Northern        504             180             35.7%
Bolga Ea

In [9]:
print(f"Total unique road types: {roads['fclass'].nunique()}\n")
for fclass in sorted(roads['fclass'].unique()):
    print(f"  {fclass}")

Total unique road types: 28

  bridleway
  busway
  cycleway
  footway
  living_street
  motorway
  motorway_link
  path
  pedestrian
  primary
  primary_link
  residential
  secondary
  secondary_link
  service
  steps
  tertiary
  tertiary_link
  track
  track_grade1
  track_grade2
  track_grade3
  track_grade4
  track_grade5
  trunk
  trunk_link
  unclassified
  unknown


In [10]:
# Ghana-corrected speed map

speed_map = {
    # Major Roads — reduced for real Ghana conditions
    'motorway': 80, 'motorway_link': 25,
    'trunk': 60, 'trunk_link': 25,
    'primary': 45, 'primary_link': 25,
    
    # Connecting Roads
    'secondary': 40, 'secondary_link': 20,
    'tertiary': 30, 'tertiary_link': 15,
    
    # Urban/Town Roads
    'residential': 25,
    'living_street': 15,
    'service': 15,
    'busway': 25,
    
    # Rural/Unpaved — kept as is, already realistic
    'unclassified': 20,
    'track': 15,
    'track_grade1': 15,
    'track_grade2': 10,
    'track_grade3': 10,
    'track_grade4': 5,
    'track_grade5': 5,
    'unknown': 15,
    
    # Walking/Non-vehicle
    'path': 4,
    'footway': 4,
    'pedestrian': 4,
    'steps': 3,
    'cycleway': 10,
    'bridleway': 4,
}

print(f"Total road types in data: {roads['fclass'].nunique()}")
print(f"Road types with speeds assigned: {len(speed_map)}")

# Check: are we covering all road types?
all_types = set(roads['fclass'].unique())
covered = set(speed_map.keys())
missing = all_types - covered

if missing:
    print(f"\n⚠️ Missing speed assignments for: {missing}")
else:
    print(f"\n✅ All {len(all_types)} road types covered!")

print(f"\n{'Road Type':<20} {'Speed (km/h)':<15} {'Category'}")
print("="*55)
for road_type, speed in speed_map.items():
    if speed >= 40:
        cat = 'Fast vehicle'
    elif speed >= 15:
        cat = 'Slow vehicle'
    elif speed >= 4:
        cat = 'Walking'
    else:
        cat = 'Very slow walking'
    print(f"{road_type:<20} {speed:<15} {cat}")

Total road types in data: 28
Road types with speeds assigned: 28

✅ All 28 road types covered!

Road Type            Speed (km/h)    Category
motorway             80              Fast vehicle
motorway_link        25              Slow vehicle
trunk                60              Fast vehicle
trunk_link           25              Slow vehicle
primary              45              Fast vehicle
primary_link         25              Slow vehicle
secondary            40              Fast vehicle
secondary_link       20              Slow vehicle
tertiary             30              Slow vehicle
tertiary_link        15              Slow vehicle
residential          25              Slow vehicle
living_street        15              Slow vehicle
service              15              Slow vehicle
busway               25              Slow vehicle
unclassified         20              Slow vehicle
track                15              Slow vehicle
track_grade1         15              Slow vehicle
track_gr

In [11]:
# 5-category transport mode for explainable 2SFCA

transport_mode = {
    # Major Roads — highways and national routes
    'motorway': 'major_road', 'motorway_link': 'major_road',
    'trunk': 'major_road', 'trunk_link': 'major_road',
    'primary': 'major_road', 'primary_link': 'major_road',
    
    # Connecting Roads — linking towns and villages
    'secondary': 'connecting_road', 'secondary_link': 'connecting_road',
    'tertiary': 'connecting_road', 'tertiary_link': 'connecting_road',
    
    # Urban/Town Roads — within settlements
    'residential': 'urban_road',
    'living_street': 'urban_road',
    'service': 'urban_road',
    'busway': 'urban_road',
    
    # Rural/Unpaved — dirt tracks, laterite, rough roads
    'unclassified': 'rural_unpaved',
    'track': 'rural_unpaved',
    'track_grade1': 'rural_unpaved',
    'track_grade2': 'rural_unpaved',
    'track_grade3': 'rural_unpaved',
    'track_grade4': 'rural_unpaved',
    'track_grade5': 'rural_unpaved',
    'cycleway': 'rural_unpaved',
    'unknown': 'rural_unpaved',
    
    # Walking — on foot only
    'path': 'walking',
    'footway': 'walking',
    'pedestrian': 'walking',
    'steps': 'walking',
    'bridleway': 'walking',
}

print(f"{'Road Type':<20} {'Speed (km/h)':<15} {'Journey Category'}")
print("="*55)
for road_type in speed_map:
    mode = transport_mode[road_type]
    print(f"{road_type:<20} {speed_map[road_type]:<15} {mode}")

# Count per category
from collections import Counter
counts = Counter(transport_mode.values())
print(f"\nRoad types per category:")
for cat, count in counts.items():
    print(f"  {cat}: {count} road types")

Road Type            Speed (km/h)    Journey Category
motorway             80              major_road
motorway_link        25              major_road
trunk                60              major_road
trunk_link           25              major_road
primary              45              major_road
primary_link         25              major_road
secondary            40              connecting_road
secondary_link       20              connecting_road
tertiary             30              connecting_road
tertiary_link        15              connecting_road
residential          25              urban_road
living_street        15              urban_road
service              15              urban_road
busway               25              urban_road
unclassified         20              rural_unpaved
track                15              rural_unpaved
track_grade1         15              rural_unpaved
track_grade2         10              rural_unpaved
track_grade3         10              rural_unpaved

In [12]:
import numpy as np
import time

# Step 1: Calculate travel time and assign transport mode for EVERY road segment

roads['speed_kmh'] = roads['fclass'].map(speed_map)
roads['transport_mode'] = roads['fclass'].map(transport_mode)

# Quick check — any roads without a speed or mode?
print(f"Roads without speed: {roads['speed_kmh'].isna().sum()}")
print(f"Roads without transport mode: {roads['transport_mode'].isna().sum()}")
print(f"Total roads: {len(roads)}")

Roads without speed: 0
Roads without transport mode: 0
Total roads: 373884


In [13]:
import networkx as nx
import time

print("Building road network graph with ALL road types...")
print("This will take a few minutes...\n")

start_time = time.time()

# Create an empty graph — this will hold all road connections
G = nx.Graph()
total_roads = len(roads)

# Loop through every road segment in Ghana
for count, (_, row) in enumerate(roads.iterrows()):
    
    # Each road is a line made up of multiple coordinate points
    coords = list(row.geometry.coords)
    speed = row['speed_kmh']       # How fast you can travel on this road
    mode = row['transport_mode']   # What category: vehicle, walking, etc.
    
    # Connect every consecutive pair of points along this road
    for i in range(len(coords) - 1):
        start = coords[i]   # Starting point (lon, lat)
        end = coords[i + 1] # Next point (lon, lat)
        
        # Calculate the real-world distance between these two points in meters
        dlat = (end[1] - start[1]) * 111320
        dlon = (end[0] - start[0]) * 111320 * np.cos(np.radians((start[1] + end[1]) / 2))
        distance_m = np.sqrt(dlat**2 + dlon**2)
        
        # Convert distance to travel time in minutes
        # Example: 2km road at 30 km/h = (2/30) * 60 = 4 minutes
        travel_time = (distance_m / 1000) / speed * 60
        
        # Add this connection to the graph with travel time and transport mode
        G.add_edge(start, end, weight=travel_time, mode=mode)
    
    # Show progress every 50,000 roads
    if (count + 1) % 50000 == 0:
        elapsed = round((time.time() - start_time) / 60, 1)
        pct = round((count + 1) / total_roads * 100, 1)
        print(f"  {count + 1:,}/{total_roads:,} roads processed ({pct}%) — {elapsed} min elapsed")

elapsed = round((time.time() - start_time) / 60, 1)

print(f"\n✅ Network built in {elapsed} minutes!")
print(f"   Nodes (intersections): {G.number_of_nodes():,}")
print(f"   Edges (road segments): {G.number_of_edges():,}")

# Check: is the network mostly connected or fragmented?
components = list(nx.connected_components(G))
component_sizes = sorted([len(c) for c in components], reverse=True)
print(f"\n   Connected components: {len(components)}")
print(f"   Largest component: {component_sizes[0]:,} nodes ({round(component_sizes[0]/G.number_of_nodes()*100, 1)}%)")

Building road network graph with ALL road types...
This will take a few minutes...

  50,000/373,884 roads processed (13.4%) — 0.8 min elapsed
  100,000/373,884 roads processed (26.7%) — 1.4 min elapsed
  150,000/373,884 roads processed (40.1%) — 2.0 min elapsed
  200,000/373,884 roads processed (53.5%) — 2.5 min elapsed
  250,000/373,884 roads processed (66.9%) — 3.0 min elapsed
  300,000/373,884 roads processed (80.2%) — 3.6 min elapsed
  350,000/373,884 roads processed (93.6%) — 4.1 min elapsed

✅ Network built in 4.3 minutes!
   Nodes (intersections): 4,374,664
   Edges (road segments): 4,542,327

   Connected components: 1180
   Largest component: 4,286,670 nodes (98.0%)


In [15]:
# Check what we have in memory
try:
    print(f"Master: {len(master)} facilities")
except:
    print("Master: NOT loaded")

try:
    print(f"Pop data: {len(pop_df)} grid cells")
except:
    print("Pop data: NOT loaded")

Master: 9978 facilities
Pop data: NOT loaded


In [16]:
import os

# Check if the processed population CSV exists
pop_csv = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\e2sfca_accessibility_scores.csv"

if os.path.exists(pop_csv):
    print("✅ Found saved population CSV from last run")
else:
    print("❌ No saved CSV — need to reload from raster")

✅ Found saved population CSV from last run


In [17]:
import rasterio

# Load population density raster
pop_raster = rasterio.open(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\gha_pd_2020_1km_UNadj.tif")
pop_data = pop_raster.read(1)

print(f"✅ Population raster loaded: {pop_data.shape} ({pop_data.shape[0]} rows x {pop_data.shape[1]} columns)")

# Extract population points — every 1km grid cell where people live
pop_points = []

for row in range(pop_data.shape[0]):
    for col in range(pop_data.shape[1]):
        pop = pop_data[row, col]
        if pop > 0:  # Only cells where people actually live
            # Convert grid position to lat/lon coordinates
            lon, lat = pop_raster.xy(row, col)
            pop_points.append({'lat': lat, 'lon': lon, 'population': pop})

pop_df = pd.DataFrame(pop_points)

print(f"✅ Population points: {len(pop_df):,} grid cells with people")
print(f"   Total population (WorldPop): {int(pop_df['population'].sum()):,}")

# Normalize to match 2021 census total
census_total = 30832019
scale_factor = census_total / pop_df['population'].sum()
pop_df['population_normalized'] = pop_df['population'] * scale_factor

print(f"\n   Census total: {census_total:,}")
print(f"   Scale factor: {round(scale_factor, 4)}")
print(f"   Normalized total: {int(pop_df['population_normalized'].sum()):,}")

✅ Population raster loaded: (773, 533) (773 rows x 533 columns)
✅ Population points: 278,001 grid cells with people
   Total population (WorldPop): 36,950,968

   Census total: 30,832,019
   Scale factor: 0.8343999981880188
   Normalized total: 30,832,016


In [18]:
from scipy.spatial import cKDTree
import time

# Build spatial index — this lets us quickly find the nearest road node to any point
print("Building spatial index for the road network...")

nodes = list(G.nodes())
node_coords = np.array(nodes)
tree = cKDTree(node_coords)

print(f"✅ Spatial index built for {len(nodes):,} road nodes")

# Snap facilities to nearest road node
print("\nSnapping facilities to road network...")
start_time = time.time()

facility_node_ids = []
for _, row in master.iterrows():
    coord = (row['Longitude'], row['Latitude'])
    _, idx = tree.query(coord)
    facility_node_ids.append(idx)

master['node_idx'] = facility_node_ids
elapsed = round(time.time() - start_time, 1)
print(f"✅ {len(facility_node_ids):,} facilities snapped ({elapsed}s)")

# Snap population points to nearest road node
print("\nSnapping population points to road network... (this may take a minute)")
start_time = time.time()

pop_node_ids = []
for _, row in pop_df.iterrows():
    coord = (row['lon'], row['lat'])
    _, idx = tree.query(coord)
    pop_node_ids.append(idx)

pop_df['node_idx'] = pop_node_ids
elapsed = round(time.time() - start_time, 1)
print(f"✅ {len(pop_node_ids):,} population points snapped ({elapsed}s)")

Building spatial index for the road network...
✅ Spatial index built for 4,374,664 road nodes

Snapping facilities to road network...
✅ 9,978 facilities snapped (3.3s)

Snapping population points to road network... (this may take a minute)
✅ 278,001 population points snapped (101.0s)


In [19]:
import igraph as ig
import time

print("Converting to igraph for faster computation...")
start_time = time.time()

# Map coordinate nodes to integer IDs
node_to_id = {node: i for i, node in enumerate(nodes)}

# Build edges and weights lists
edges = []
weights = []
modes = []

for (start, end, data) in G.edges(data=True):
    edges.append((node_to_id[start], node_to_id[end]))
    weights.append(data['weight'])
    modes.append(data['mode'])

# Create igraph network
g_ig = ig.Graph(n=len(node_to_id), edges=edges, directed=False)
g_ig.es['weight'] = weights
g_ig.es['mode'] = modes  # Transport mode on every edge for explainability

elapsed = round((time.time() - start_time) / 60, 1)

print(f"✅ igraph network built in {elapsed} min!")
print(f"   Nodes: {g_ig.vcount():,}")
print(f"   Edges: {g_ig.ecount():,}")

# Update facility and population node IDs to match igraph
master['node_id'] = master['node_idx'].apply(lambda x: node_to_id[nodes[x]])
pop_df['node_id'] = pop_df['node_idx'].apply(lambda x: node_to_id[nodes[x]])

print(f"\n✅ Facilities and population points mapped to igraph IDs")

Converting to igraph for faster computation...
✅ igraph network built in 0.5 min!
   Nodes: 4,374,664
   Edges: 4,542,327

✅ Facilities and population points mapped to igraph IDs


In [20]:
import time

# Quick test — calculate distances from one facility to all nodes
test_facility_id = master.iloc[0]['node_id']

start_time = time.time()
distances = g_ig.distances(source=test_facility_id, weights='weight')[0]
elapsed = time.time() - start_time

# How many nodes within our 120-minute cutoff?
within_120 = sum(1 for d in distances if d <= 120)
within_60 = sum(1 for d in distances if d <= 60)

print(f"Test facility: {master.iloc[0]['Name']}")
print(f"Calculation time: {round(elapsed, 2)} seconds")
print(f"\nNodes reachable within 60 min: {within_60:,}")
print(f"Nodes reachable within 120 min: {within_120:,}")

# Estimate total computation time
print(f"\nEstimated time for all {len(master):,} facilities:")
print(f"  One at a time: {round(elapsed * len(master) / 3600, 1)} hours")
print(f"  Batched (50): {round(elapsed * len(master) / 50 / 3600 * 50, 1)} hours")

Test facility: 1 MEDICAL RECEPTION STATION(1MRS)
Calculation time: 4.93 seconds

Nodes reachable within 60 min: 564,920
Nodes reachable within 120 min: 968,509

Estimated time for all 9,978 facilities:
  One at a time: 13.7 hours
  Batched (50): 13.7 hours


In [21]:
# Test actual batch of 50
start_time = time.time()

test_ids = []
for i in range(50):
    test_ids.append(master.iloc[i]['node_id'])

results = g_ig.distances(source=test_ids, weights='weight')
elapsed = time.time() - start_time

print(f"50 facilities batched: {round(elapsed, 1)} seconds")
print(f"Per facility: {round(elapsed/50, 2)} seconds")
print(f"\nEstimated time for all {len(master):,} facilities:")
print(f"  {round(elapsed/50 * len(master) / 3600, 1)} hours")

50 facilities batched: 133.8 seconds
Per facility: 2.68 seconds

Estimated time for all 9,978 facilities:
  7.4 hours


In [22]:
# Build population lookup
print("Building population lookup...")
pop_node_lookup = {}
for i, row in pop_df.iterrows():
    nid = row['node_id']
    if nid not in pop_node_lookup:
        pop_node_lookup[nid] = []
    pop_node_lookup[nid].append((i, row['population_normalized']))

print(f"✅ Population lookup built — {len(pop_node_lookup):,} unique nodes with population")

Building population lookup...
✅ Population lookup built — 174,930 unique nodes with population


In [26]:
import pickle
import time
import numpy as np

# ============================================================
# JOB 1: ENHANCED TWO-STEP FLOATING CATCHMENT AREA (E2SFCA)
# 
# What this does:
#   For every 1km grid cell in Ghana where people live,
#   calculate an accessibility score that considers:
#   1. How many facilities can people there reach within 120 min?
#   2. How crowded are those facilities?
#   3. How far away are they? (closer = counts more)
#
# Higher score = better access to healthcare
# Lower score = worse access to healthcare
# Zero score = no facility reachable within 120 minutes
# ============================================================

# --- PARAMETERS ---
CUTOFF = 120        # Maximum travel time in minutes (extended for rural Ghana)
BATCH_SIZE = 50     # Process 50 facilities at a time for speed

# --- DECAY FUNCTION ---
# A facility 5 minutes away is more useful than one 90 minutes away
# This function assigns a weight based on how far the facility is
def decay_weight(travel_time):
    if travel_time <= 15:
        return 1.0    # 0-15 min: Immediate access, fully counts
    elif travel_time <= 30:
        return 0.8    # 16-30 min: Good access, mostly counts
    elif travel_time <= 60:
        return 0.5    # 31-60 min: Moderate access, half counts
    elif travel_time <= 120:
        return 0.2    # 61-120 min: Marginalized access, barely counts
    else:
        return 0.0    # Over 2 hours: No access

# --- INITIALIZE RESULTS ---
facility_ratios = {}
accessibility_scores = np.zeros(len(pop_df))

# --- HOW MANY BATCHES DO WE NEED? ---
total_batches = (len(master) + BATCH_SIZE - 1) // BATCH_SIZE
start_time = time.time()

print(f"{'='*60}")
print(f"JOB 1: E2SFCA ACCESSIBILITY SCORES")
print(f"{'='*60}")
print(f"Facilities: {len(master):,}")
print(f"Population points: {len(pop_df):,}")
print(f"Cutoff: {CUTOFF} minutes")
print(f"Decay: 1.0 (0-15min) → 0.8 (16-30) → 0.5 (31-60) → 0.2 (61-120) → 0.0")
print(f"Batches: {total_batches} (size {BATCH_SIZE})")
print(f"Estimated time: ~7 hours")
print(f"{'='*60}\n")

# --- MAIN LOOP: PROCESS FACILITIES IN BATCHES ---
for batch_num in range(total_batches):
    
    batch_start_time = time.time()
    
    # Which facilities are in this batch?
    batch_start = batch_num * BATCH_SIZE
    batch_end = min(batch_start + BATCH_SIZE, len(master))
    
    # Get the road network node IDs for this batch of facilities
    batch_node_ids = master.iloc[batch_start:batch_end]['node_id'].tolist()
    
    # Calculate travel time from every facility in this batch to EVERY node
    results = g_ig.distances(source=batch_node_ids, weights='weight')
    
    # Track how many population points are within catchment for this batch
    batch_catchment_count = 0
    
    # Process each facility in the batch
    for i, facility_idx in enumerate(range(batch_start, batch_end)):
        distances = results[i]
        
        # STEP 1: How busy is this facility?
        weighted_pop = 0
        catchment_pops = []
        
        for nid in range(len(distances)):
            if distances[nid] <= CUTOFF:
                w = decay_weight(distances[nid])
                if w > 0 and nid in pop_node_lookup:
                    for pop_idx, pop in pop_node_lookup[nid]:
                        weighted_pop += pop * w
                        catchment_pops.append((pop_idx, w))
        
        ratio = 1.0 / weighted_pop if weighted_pop > 0 else 0
        facility_ratios[facility_idx] = ratio
        batch_catchment_count += len(catchment_pops)
        
        # STEP 2: Distribute this facility's ratio to population points
        for pop_idx, w in catchment_pops:
            accessibility_scores[pop_idx] += ratio * w
    
    # --- PROGRESS UPDATE every single batch ---
    elapsed = time.time() - start_time
    batch_time = round(time.time() - batch_start_time, 1)
    pct = (batch_num + 1) / total_batches
    eta_hours = (elapsed / pct - elapsed) / 3600
    facilities_done = batch_end
    scored_so_far = (accessibility_scores > 0).sum()
    
    print(f"  Batch {batch_num+1}/{total_batches} "
          f"({round(pct*100,1)}%) | "
          f"Facilities: {facilities_done:,}/{len(master):,} | "
          f"Scored: {scored_so_far:,} pop points | "
          f"Batch time: {batch_time}s | "
          f"ETA: {round(eta_hours, 1)}h remaining")
    
    # --- SAVE PROGRESS every 25 batches ---
    if (batch_num + 1) % 25 == 0:
        save_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\e2sfca_v2_progress.pkl"
        with open(save_path, 'wb') as f:
            pickle.dump({
                'facility_ratios': facility_ratios,
                'accessibility_scores': accessibility_scores,
                'last_batch': batch_num
            }, f)
        print(f"  💾 Progress saved at batch {batch_num+1}")

# --- FINAL SAVE ---
save_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\e2sfca_v2_complete.pkl"
with open(save_path, 'wb') as f:
    pickle.dump({
        'facility_ratios': facility_ratios,
        'accessibility_scores': accessibility_scores
    }, f)

total_time = (time.time() - start_time) / 3600
print(f"\n{'='*60}")
print(f"✅ JOB 1 COMPLETE!")
print(f"{'='*60}")
print(f"Total time: {round(total_time, 1)} hours")
print(f"Facilities processed: {len(facility_ratios):,}")
print(f"Population points with access: {(accessibility_scores > 0).sum():,}")
print(f"Population points with NO access: {(accessibility_scores == 0).sum():,}")

JOB 1: E2SFCA ACCESSIBILITY SCORES
Facilities: 9,978
Population points: 278,001
Cutoff: 120 minutes
Decay: 1.0 (0-15min) → 0.8 (16-30) → 0.5 (31-60) → 0.2 (61-120) → 0.0
Batches: 200 (size 50)
Estimated time: ~7 hours

  Batch 1/200 (0.5%) | Facilities: 50/9,978 | Scored: 126,537 pop points | Batch time: 268.1s | ETA: 14.8h remaining
  Batch 2/200 (1.0%) | Facilities: 100/9,978 | Scored: 153,739 pop points | Batch time: 257.3s | ETA: 14.4h remaining
  Batch 3/200 (1.5%) | Facilities: 150/9,978 | Scored: 157,219 pop points | Batch time: 229.1s | ETA: 13.8h remaining
  Batch 4/200 (2.0%) | Facilities: 200/9,978 | Scored: 167,347 pop points | Batch time: 263.4s | ETA: 13.9h remaining
  Batch 5/200 (2.5%) | Facilities: 250/9,978 | Scored: 175,076 pop points | Batch time: 250.9s | ETA: 13.7h remaining
  Batch 6/200 (3.0%) | Facilities: 300/9,978 | Scored: 175,812 pop points | Batch time: 261.2s | ETA: 13.7h remaining
  Batch 7/200 (3.5%) | Facilities: 350/9,978 | Scored: 183,686 pop points 

In [27]:
# Save results as CSV
pop_df['accessibility_score'] = accessibility_scores

pop_df.to_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\e2sfca_v2_scores.csv", index=False)

print(f"✅ Saved as CSV: {len(pop_df):,} population points with scores")
print(f"   Points with access: {(pop_df['accessibility_score'] > 0).sum():,}")
print(f"   Points with NO access: {(pop_df['accessibility_score'] == 0).sum():,}")

✅ Saved as CSV: 278,001 population points with scores
   Points with access: 268,347
   Points with NO access: 9,654


In [28]:
# Quick overview of the results
print("=== E2SFCA V2 RESULTS OVERVIEW ===\n")

scored = pop_df[pop_df['accessibility_score'] > 0]

print(f"Population points with access: {len(scored):,} ({round(len(scored)/len(pop_df)*100, 1)}%)")
print(f"Population points with NO access: {(pop_df['accessibility_score'] == 0).sum():,} ({round((pop_df['accessibility_score'] == 0).sum()/len(pop_df)*100, 1)}%)")

# How many PEOPLE have no access?
no_access_pop = pop_df[pop_df['accessibility_score'] == 0]['population_normalized'].sum()
total_pop = pop_df['population_normalized'].sum()
print(f"\nPeople with NO access within 120 min: {int(no_access_pop):,} ({round(no_access_pop/total_pop*100, 1)}%)")
print(f"People WITH access: {int(total_pop - no_access_pop):,} ({round((total_pop - no_access_pop)/total_pop*100, 1)}%)")

print(f"\nAccessibility score stats:")
print(scored['accessibility_score'].describe())

=== E2SFCA V2 RESULTS OVERVIEW ===

Population points with access: 268,347 (96.5%)
Population points with NO access: 9,654 (3.5%)

People with NO access within 120 min: 210,680 (0.699999988079071%)
People WITH access: 30,621,336 (99.30000305175781%)

Accessibility score stats:
count    268347.000000
mean          0.000281
std           0.000177
min           0.000002
25%           0.000171
50%           0.000249
75%           0.000354
max           0.022834
Name: accessibility_score, dtype: float64


In [29]:
# Check what's still in memory
print("Checking what's loaded...\n")

# Check the graph
try:
    print(f"✅ Graph loaded: {g.vcount():,} nodes, {g.ecount():,} edges")
except:
    print("❌ Graph (g) not found")

# Check population data
try:
    print(f"✅ Population data loaded: {len(pop_df):,} rows")
except:
    print("❌ pop_df not found")

# Check master facility data
try:
    print(f"✅ Master dataset loaded: {len(master):,} facilities")
except:
    print("❌ master not found")

# Check E2SFCA scores
try:
    print(f"✅ E2SFCA scores loaded: {len(pop_df['accessibility_score'].dropna()):,} scored points")
except:
    print("❌ accessibility_score not found")

# Check Facility_Tier exists
try:
    print(f"✅ Facility_Tier column exists: {master['Facility_Tier'].value_counts().to_dict()}")
except:
    print("❌ Facility_Tier not found")

Checking what's loaded...

❌ Graph (g) not found
✅ Population data loaded: 278,001 rows
✅ Master dataset loaded: 9,978 facilities
✅ E2SFCA scores loaded: 278,001 scored points
✅ Facility_Tier column exists: {'Basic': 6732, 'Mid': 2392, 'Advanced': 854}


In [30]:
import igraph as ig
import geopandas as gpd
import pandas as pd
import numpy as np
import time

print("Rebuilding road network graph...")
start = time.time()

# --- LOAD THE ROAD SHAPEFILE ---
roads = gpd.read_file(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\ghana-260322-free.shp\gis_osm_roads_free_1.shp")
print(f"✅ Roads loaded: {len(roads):,} segments")

# --- SPEED LOOKUP TABLE ---
# Same speeds we used in the original computation
speed_map = {
    'motorway': 80, 'motorway_link': 25, 'trunk': 60, 'trunk_link': 25,
    'primary': 45, 'primary_link': 25, 'secondary': 40, 'secondary_link': 20,
    'tertiary': 30, 'tertiary_link': 15, 'residential': 25, 'living_street': 15,
    'service': 15, 'busway': 25, 'unclassified': 20, 'track': 15,
    'track_grade1': 15, 'track_grade2': 10, 'track_grade3': 10,
    'track_grade4': 5, 'track_grade5': 5, 'cycleway': 10, 'unknown': 15,
    'path': 4, 'footway': 4, 'pedestrian': 4, 'steps': 3, 'bridleway': 4
}

# --- MODE LOOKUP TABLE ---
mode_map = {
    'motorway': 'major_road', 'motorway_link': 'major_road',
    'trunk': 'major_road', 'trunk_link': 'major_road',
    'primary': 'major_road', 'primary_link': 'major_road',
    'secondary': 'connecting_road', 'secondary_link': 'connecting_road',
    'tertiary': 'connecting_road', 'tertiary_link': 'connecting_road',
    'residential': 'urban_road', 'living_street': 'urban_road',
    'service': 'urban_road', 'busway': 'urban_road',
    'unclassified': 'rural_unpaved', 'track': 'rural_unpaved',
    'track_grade1': 'rural_unpaved', 'track_grade2': 'rural_unpaved',
    'track_grade3': 'rural_unpaved', 'track_grade4': 'rural_unpaved',
    'track_grade5': 'rural_unpaved', 'cycleway': 'rural_unpaved',
    'unknown': 'rural_unpaved', 'path': 'walking', 'footway': 'walking',
    'pedestrian': 'walking', 'steps': 'walking', 'bridleway': 'walking'
}

# --- EXTRACT ALL NODES (coordinate points) FROM ROAD SEGMENTS ---
print("Extracting nodes from road segments...")
node_coords = {}   # maps (lon, lat) → node index
edges_list = []    # list of (node_a, node_b, travel_time, mode)

node_counter = 0

for _, row in roads.iterrows():
    road_type = row.get('fclass', 'unknown')
    speed = speed_map.get(road_type, 15)  # default 15 if unknown
    mode = mode_map.get(road_type, 'rural_unpaved')
    
    # Get all coordinate points along this road segment
    if row.geometry is None:
        continue
    coords = list(row.geometry.coords)
    
    # Register each point as a node if not already seen
    segment_nodes = []
    for coord in coords:
        if coord not in node_coords:
            node_coords[coord] = node_counter
            node_counter += 1
        segment_nodes.append(node_coords[coord])
    
    # Create edges between consecutive points
    for i in range(len(segment_nodes) - 1):
        a = segment_nodes[i]
        b = segment_nodes[i + 1]
        
        # Calculate distance between points in km (approximate)
        lon1, lat1 = coords[i]
        lon2, lat2 = coords[i + 1]
        dlat = np.radians(lat2 - lat1)
        dlon = np.radians(lon2 - lon1)
        a_val = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
        dist_km = 6371 * 2 * np.arcsin(np.sqrt(a_val))
        
        # Travel time in minutes
        travel_time = (dist_km / speed) * 60
        
        edges_list.append((a, b, travel_time, mode))

print(f"✅ Nodes extracted: {node_counter:,}")
print(f"✅ Edges extracted: {len(edges_list):,}")

# --- BUILD IGRAPH GRAPH ---
print("Building igraph graph...")
g = ig.Graph()
g.add_vertices(node_counter)

# Store coordinates as vertex attributes
coords_list = [''] * node_counter
for coord, idx in node_coords.items():
    coords_list[idx] = coord
g.vs['coord'] = coords_list

# Add edges with travel time and mode
g.add_edges([(e[0], e[1]) for e in edges_list])
g.es['weight'] = [e[2] for e in edges_list]
g.es['mode'] = [e[3] for e in edges_list]

# --- KEEP ONLY LARGEST CONNECTED COMPONENT ---
print("Finding largest connected component...")
components = g.clusters()
largest = components.giant()
g = largest
print(f"✅ Graph built: {g.vcount():,} nodes, {g.ecount():,} edges")

elapsed = round((time.time() - start) / 60, 1)
print(f"\n✅ Done! Took {elapsed} minutes")

Rebuilding road network graph...
✅ Roads loaded: 373,884 segments
Extracting nodes from road segments...
✅ Nodes extracted: 4,374,664
✅ Edges extracted: 4,542,352
Building igraph graph...
Finding largest connected component...


C:\Users\hp\AppData\Local\Temp\ipykernel_24448\1373045847.py:107: DeprecationWarning: Graph.clusters() is deprecated; use Graph.connected_components() instead
  components = g.clusters()


✅ Graph built: 4,286,670 nodes, 4,454,860 edges

✅ Done! Took 3.6 minutes


In [31]:
from scipy.spatial import cKDTree

print("Setting up coordinate lookup...\n")

# --- EXTRACT ALL NODE COORDINATES FROM GRAPH ---
# These are all the road intersection/point coordinates
all_coords = np.array(g.vs['coord'])  # shape: (4,286,670 x 2) — (lon, lat)
lon_lat = np.array([[c[0], c[1]] for c in all_coords])

# --- BUILD A SPATIAL INDEX (KD-TREE) ---
# This lets us find the nearest road node to any point in milliseconds
print("Building KD-Tree spatial index...")
tree = cKDTree(lon_lat)
print(f"✅ KD-Tree built on {len(lon_lat):,} road nodes")

# --- SNAP FACILITIES TO NEAREST ROAD NODE ---
print("\nSnapping facilities to road network...")
facility_coords = master[['Longitude', 'Latitude']].values
_, facility_node_ids = tree.query(facility_coords, k=1)
master['node_id'] = facility_node_ids
print(f"✅ {len(master):,} facilities snapped to road nodes")

# --- SNAP POPULATION POINTS TO NEAREST ROAD NODE ---
print("\nSnapping population points to road network...")
pop_coords = pop_df[['lon', 'lat']].values
_, pop_node_ids = tree.query(pop_coords, k=1)
pop_df['node_id'] = pop_node_ids
print(f"✅ {len(pop_df):,} population points snapped to road nodes")

print("\n✅ All done! Ready for travel time computation.")

Setting up coordinate lookup...

Building KD-Tree spatial index...
✅ KD-Tree built on 4,286,670 road nodes

Snapping facilities to road network...
✅ 9,978 facilities snapped to road nodes

Snapping population points to road network...
✅ 278,001 population points snapped to road nodes

✅ All done! Ready for travel time computation.


In [32]:
# Let's see where the psychiatric hospitals and leprosaria are

print("=== PSYCHIATRIC HOSPITALS ===\n")
psychiatric = master[master['Facility_Type'] == 'PSYCHIATRIC HOSPITAL'][['Name', 'Region', 'District', 'Ownership', 'Latitude', 'Longitude']]
print(psychiatric.to_string(index=False))

print("\n\n=== LEPROSARIA ===\n")
leprosarium = master[master['Facility_Type'] == 'LEPROSARIUM'][['Name', 'Region', 'District', 'Ownership', 'Latitude', 'Longitude']]
print(leprosarium.to_string(index=False))

=== PSYCHIATRIC HOSPITALS ===

                                       Name        Region                     District  Ownership  Latitude  Longitude
                 ACCRA PSYCHIATRIC HOSPITAL GREATER ACCRA                KORLE-KLOTTEY GOVERNMENT  5.562669  -0.205186
                ANKAFUL PSYCHIATRY HOSPITAL       CENTRAL KOMENDA-EDINA-EGUAFO-ABIREM- GOVERNMENT  5.154714  -1.322669
                           PANTANG HOSPITAL GREATER ACCRA        LA-NKWANTANANG-MADINA GOVERNMENT  5.714847  -0.187670
PASSION FOR TOTAL CARE MENTAL HEALTH CLINIC    UPPER EAST                      TEMPANE    PRIVATE 10.990202  -0.039952
                PEACE BE CONSULTANCY CLINIC GREATER ACCRA                       ADENTA    PRIVATE  5.684570  -0.145158


=== LEPROSARIA ===

                        Name  Region                     District  Ownership  Latitude  Longitude
ANKAFUL GEN LEPROSY HOSPITAL CENTRAL KOMENDA-EDINA-EGUAFO-ABIREM- GOVERNMENT  5.151288  -1.317994
       BEPOASE NEW TOWN CHPS ASHANTI 

In [33]:
# Let's look at the leprosarium that seems suspicious
print(master[master['Name'] == 'BEPOASE NEW TOWN CHPS'].to_string())

        ID                   Name Facility_Type Ownership   Region       District Sub-District Community  Latitude  Longitude  has_emonc  has_midwife  has_blood_bank  Centroid_Lat  Centroid_Lon  distance_from_centroid_km  District_Population  Male_Total  Female_Total  Household_Total NonHousehold_Total Urban_Population Urban_Male Urban_Female Rural_Population Rural_Male Rural_Female  Percentage of Urban  Percentage of Rural Facility_Tier  node_idx  node_id
2268  8895  BEPOASE NEW TOWN CHPS   LEPROSARIUM      CHAG  ASHANTI  SEKYERE SOUTH     WIAMOASE   BEPOASE  7.086488  -1.555215      False        False           False      6.986597     -1.517094                     11.877               120076       58065         62011           113253               6823            87553      41883        45670            32523      16182        16341             0.729147             0.270853      Advanced   2017194  1973109


In [34]:
# Correct the Bepoase New Town CHPS facility type
# It is clearly a CHPS compound, not a Leprosarium

print("Before correction:")
print(master[master['ID'] == 8895][['Name', 'Facility_Type', 'Facility_Tier']])

# Fix the Facility_Type and Facility_Tier
master.loc[master['ID'] == 8895, 'Facility_Type'] = 'CHPS'
master.loc[master['ID'] == 8895, 'Facility_Tier'] = 'Basic'

print("\nAfter correction:")
print(master[master['ID'] == 8895][['Name', 'Facility_Type', 'Facility_Tier']])

# Verify the leprosarium count now
print("\n=== LEPROSARIA (after correction) ===")
print(master[master['Facility_Type'] == 'LEPROSARIUM'][['Name', 'Region', 'District']])

# Save the corrected master dataset
master.to_csv(
    r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv",
    index=False
)

print("\n✅ Correction saved to master_dataset_v3.csv")

Before correction:
                       Name Facility_Type Facility_Tier
2268  BEPOASE NEW TOWN CHPS   LEPROSARIUM      Advanced

After correction:
                       Name Facility_Type Facility_Tier
2268  BEPOASE NEW TOWN CHPS          CHPS         Basic

=== LEPROSARIA (after correction) ===
                              Name   Region                      District
1244  ANKAFUL GEN LEPROSY HOSPITAL  CENTRAL  KOMENDA-EDINA-EGUAFO-ABIREM-

✅ Correction saved to master_dataset_v3.csv


In [35]:
# Confirm our facility types and counts before we start
print("=== FACILITY TYPES (after Bepoase correction) ===\n")
print(master['Facility_Type'].value_counts().to_string())

print("\n\n=== FACILITY TIER COUNTS ===\n")
print(master['Facility_Tier'].value_counts().to_string())

print("\n\n=== QUICK SANITY CHECK ===")
print(f"Total facilities: {len(master):,}")
print(f"Psychiatric hospitals: {len(master[master['Facility_Type'] == 'PSYCHIATRIC HOSPITAL']):,}")
print(f"Leprosaria: {len(master[master['Facility_Type'] == 'LEPROSARIUM']):,}")

=== FACILITY TYPES (after Bepoase correction) ===

Facility_Type
CHPS                          6733
HEALTH CENTRE                 1215
CLINIC                         929
HOSPITAL                       581
MATERNITY HOME                 248
DISTRICT HOSPITAL              145
POLYCLINIC                      93
TEACHING HOSPITAL               10
REGIONAL HOSPITAL               10
UNIVERSITY HOSPITAL/CLINIC       8
PSYCHIATRIC HOSPITAL             5
LEPROSARIUM                      1


=== FACILITY TIER COUNTS ===

Facility_Tier
Basic       6733
Mid         2392
Advanced     853


=== QUICK SANITY CHECK ===
Total facilities: 9,978
Psychiatric hospitals: 5
Leprosaria: 1


In [39]:
from scipy.spatial import cKDTree
import numpy as np

print("Testing k-d tree candidate selection...\n")

# Build a k-d tree for EMERGENCY facilities only
emergency_mask = master['Facility_Type'].isin(['HOSPITAL', 'DISTRICT HOSPITAL'])
emergency_coords = master[emergency_mask][['Longitude', 'Latitude']].values
emergency_indices = master[emergency_mask].index.values

emergency_tree = cKDTree(emergency_coords)

# For a sample of 1000 population points, find nearest 10 facilities
# by straight line distance
sample_pop_coords = pop_df[['lon', 'lat']].values[:1000]

# Query k nearest neighbors (k=10)
distances_km, neighbor_idx = emergency_tree.query(sample_pop_coords, k=10)

# Convert straight line distances to km (approximate)
distances_km = distances_km * 111  # 1 degree ≈ 111 km

print("Straight line distance to nearest emergency facility:")
print(f"  Nearest (k=1):  avg {distances_km[:,0].mean():.1f} km, max {distances_km[:,0].max():.1f} km")
print(f"  2nd nearest:    avg {distances_km[:,1].mean():.1f} km, max {distances_km[:,1].max():.1f} km")
print(f"  5th nearest:    avg {distances_km[:,4].mean():.1f} km, max {distances_km[:,4].max():.1f} km")
print(f"  10th nearest:   avg {distances_km[:,9].mean():.1f} km, max {distances_km[:,9].max():.1f} km")

Testing k-d tree candidate selection...

Straight line distance to nearest emergency facility:
  Nearest (k=1):  avg 8.9 km, max 45.5 km
  2nd nearest:    avg 13.4 km, max 49.9 km
  5th nearest:    avg 21.7 km, max 73.7 km
  10th nearest:   avg 45.3 km, max 110.5 km


In [41]:
import time

print("Speed test — batching facilities...\n")

# Take 50 emergency facility nodes as sources
test_sources = list(set(groupings['emergency'].index[:50].map(
    lambda i: master.loc[i, 'node_id']
)))

# Take 1000 unique population nodes as targets  
test_targets = unique_pop_nodes[:1000].tolist()

start = time.time()
distances = g.distances(
    source=test_sources,
    target=test_targets,
    weights='weight'
)
elapsed = round(time.time() - start, 2)

print(f"50 facilities → 1,000 pop points: {elapsed} seconds")
print(f"Extrapolated for 726 emergency facilities → 172,611 pop nodes: {round(elapsed * (726/50) * (172611/1000) / 60, 1)} minutes")

Speed test — batching facilities...

50 facilities → 1,000 pop points: 110.5 seconds
Extrapolated for 726 emergency facilities → 172,611 pop nodes: 4615.8 minutes


In [42]:
# Okay that is 3 days just for emergency facilities. That is not happening.

In [43]:
import pickle

print("Loading E2SFCA pickle...")

with open(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\e2sfca_v2_complete.pkl", 'rb') as f:
    e2sfca_data = pickle.load(f)

# See what's inside
print(f"Type: {type(e2sfca_data)}")

if isinstance(e2sfca_data, dict):
    print(f"Keys: {list(e2sfca_data.keys())}")
elif hasattr(e2sfca_data, 'columns'):
    print(f"Columns: {list(e2sfca_data.columns)}")
    print(f"Shape: {e2sfca_data.shape}")
else:
    print(e2sfca_data)

Loading E2SFCA pickle...
Type: <class 'dict'>
Keys: ['facility_ratios', 'accessibility_scores']


In [45]:
import time
import numpy as np
from scipy.spatial import cKDTree

print("SPEED TEST PER GROUPING\n")

# We will test with a sample of 5,000 population points
# and extrapolate to full 278,001
SAMPLE_SIZE = 5000
sample_coords = pop_df[['lon', 'lat']].values[:SAMPLE_SIZE]
sample_node_ids = pop_df['node_id'].values[:SAMPLE_SIZE]

def speed_test(facility_df, label):
    fac_coords = facility_df[['Longitude', 'Latitude']].values
    fac_node_ids = facility_df['node_id'].values
    fac_tree = cKDTree(fac_coords)
    
    k = min(10, len(facility_df))
    _, neighbor_idx = fac_tree.query(sample_coords, k=k)
    if k == 1:
        neighbor_idx = neighbor_idx.reshape(-1, 1)
    
    start = time.time()
    
    for i in range(SAMPLE_SIZE):
        pop_node = int(sample_node_ids[i])
        candidate_nodes = list(set([int(fac_node_ids[j]) for j in neighbor_idx[i]]))
        distances = g.distances(
            source=pop_node,
            target=candidate_nodes,
            weights='weight'
        )[0]
    
    elapsed = time.time() - start
    estimated_hours = round((elapsed / SAMPLE_SIZE) * 278001 / 3600, 1)
    print(f"  {label:<15}: {estimated_hours} hours for full dataset")

# Test each grouping
speed_test(master, "any")
speed_test(master[master['Facility_Type'] == 'PSYCHIATRIC HOSPITAL'], "psychiatric")
speed_test(master[master['Facility_Type'].isin(['HOSPITAL', 'DISTRICT HOSPITAL'])], "emergency")
speed_test(master[master['Facility_Type'].isin(['REGIONAL HOSPITAL', 'TEACHING HOSPITAL', 'UNIVERSITY HOSPITAL/CLINIC'])], "specialist")

print("\nDone! Now we know what we are getting into.")

SPEED TEST PER GROUPING

  any            : 29.8 hours for full dataset


KeyboardInterrupt: 

In [46]:
import os

# Search for the files across common locations
search_paths = [
    r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project",
    r"C:\Users\hp\Downloads",
    r"C:\Users\hp\Downloads\Code & Scripts",
    r"C:\Users\hp\Documents",
    r"C:\Users\hp\Desktop",
]

files_to_find = [
    'nearest_facility_times.csv',
    'gis_osm_roads_free_1.shp',
    'master_dataset_v3.csv',
    'e2sfca_v2_scores.csv',
]

print("Searching for files...\n")

for filename in files_to_find:
    found = False
    for path in search_paths:
        full_path = os.path.join(path, filename)
        if os.path.exists(full_path):
            size = round(os.path.getsize(full_path) / (1024*1024), 1)
            print(f"✅ {filename}")
            print(f"   Location: {full_path}")
            print(f"   Size: {size} MB\n")
            found = True
            break
    if not found:
        print(f"❌ {filename} — NOT FOUND\n")

Searching for files...

❌ nearest_facility_times.csv — NOT FOUND

❌ gis_osm_roads_free_1.shp — NOT FOUND

✅ master_dataset_v3.csv
   Location: C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv
   Size: 2.6 MB

✅ e2sfca_v2_scores.csv
   Location: C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\e2sfca_v2_scores.csv
   Size: 25.2 MB



In [47]:
import os

print("Searching entire computer for road shapefile...\n")

# Search more broadly
search_roots = [
    r"C:\Users\hp",
]

target = 'gis_osm_roads_free_1.shp'

for root_path in search_roots:
    for dirpath, dirnames, filenames in os.walk(root_path):
        # Skip hidden and system folders to speed up search
        dirnames[:] = [d for d in dirnames if not d.startswith('.') 
                      and d not in ['AppData', 'Windows', 'Program Files']]
        
        for filename in filenames:
            if filename == target:
                full_path = os.path.join(dirpath, filename)
                size = round(os.path.getsize(full_path) / (1024*1024), 1)
                print(f"✅ Found: {full_path}")
                print(f"   Size: {size} MB")

Searching entire computer for road shapefile...

✅ Found: C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\ghana-260322-free.shp\gis_osm_roads_free_1.shp
   Size: 95.0 MB


In [ ]:
# What roads do we have and how should we assign speeds?
road_counts = roads['fclass'].value_counts()

print("Road types and suggested travel speeds:\n")
speed_map = {
    'motorway': 100, 'motorway_link': 60,
    'trunk': 80, 'trunk_link': 50,
    'primary': 60, 'primary_link': 40,
    'secondary': 50, 'secondary_link': 30,
    'tertiary': 40, 'tertiary_link': 20,
    'residential': 30,
    'unclassified': 20,
    'service': 20,
    'living_street': 15,
    'track': 15,
    'track_grade1': 15, 'track_grade2': 10, 'track_grade3': 10,
    'track_grade4': 5, 'track_grade5': 5,
}

print(f"{'Road Type':<20} {'Count':<10} {'Speed (km/h)'}")
print("="*45)
for road_type, count in road_counts.items():
    speed = speed_map.get(road_type, None)
    if speed:
        print(f"{road_type:<20} {count:<10} {speed}")
    else:
        print(f"{road_type:<20} {count:<10} ---")

In [ ]:
# Step 1: Filter to driveable roads only and assign speeds

speed_map = {
    'motorway': 100, 'motorway_link': 60,
    'trunk': 80, 'trunk_link': 50,
    'primary': 60, 'primary_link': 40,
    'secondary': 50, 'secondary_link': 30,
    'tertiary': 40, 'tertiary_link': 20,
    'residential': 30,
    'unclassified': 20,
    'service': 20,
    'living_street': 15,
    'track': 15,
    'track_grade1': 15, 'track_grade2': 10, 'track_grade3': 10,
    'track_grade4': 5, 'track_grade5': 5,
    'busway': 30, 'unknown': 15,
}

# Keep only driveable roads
driveable = roads[roads['fclass'].isin(speed_map.keys())].copy()
driveable['speed_kmh'] = driveable['fclass'].map(speed_map)

# Calculate travel time in minutes for each road segment
# Length in meters (from geometry) / speed in km/h -> minutes
driveable['length_m'] = driveable.geometry.length * 111320  # rough degrees to meters at Ghana's latitude
driveable['travel_time_min'] = (driveable['length_m'] / 1000) / driveable['speed_kmh'] * 60

print(f"Driveable roads: {len(driveable)} of {len(roads)}")
print(f"Excluded: {len(roads) - len(driveable)} (paths, footways, steps, etc.)")
print(f"\nTravel time stats (minutes per segment):")
print(driveable['travel_time_min'].describe())

In [ ]:
import networkx as nx

# Step 2: Build the road network graph
print("Building road network graph... (this may take a few minutes)")

G = nx.Graph()

for _, row in driveable.iterrows():
    coords = list(row.geometry.coords)
    start = coords[0]
    end = coords[-1]
    
    # Add edge with travel time as weight
    G.add_edge(start, end, weight=row['travel_time_min'])

print(f"✅ Network built!")
print(f"   Nodes (intersections): {G.number_of_nodes():,}")
print(f"   Edges (road segments): {G.number_of_edges():,}")

In [ ]:
# Step 3: Extract population points from the raster grid

pop_points = []

for row in range(pop_data.shape[0]):
    for col in range(pop_data.shape[1]):
        pop = pop_data[row, col]
        if pop > 0:  # only squares where people live
            # Convert grid position to lat/lon coordinates
            lon, lat = pop_raster.xy(row, col)
            pop_points.append({'lat': lat, 'lon': lon, 'population': pop})

pop_df = pd.DataFrame(pop_points)

print(f"✅ Population points: {len(pop_df):,} grid cells with people")
print(f"   Total population: {int(pop_df['population'].sum()):,}")
print(f"\nPopulation per cell stats:")
print(pop_df['population'].describe())

In [ ]:
# Normalize WorldPop population to match 2021 census total

census_total = 30832019  # 2021 Ghana census total
worldpop_total = pop_df['population'].sum()

scale_factor = census_total / worldpop_total

pop_df['population_normalized'] = pop_df['population'] * scale_factor

print(f"WorldPop total: {int(worldpop_total):,}")
print(f"Census total: {census_total:,}")
print(f"Scale factor: {round(scale_factor, 4)}")
print(f"\nNormalized total: {int(pop_df['population_normalized'].sum()):,}")
print(f"\nNormalized population per cell stats:")
print(pop_df['population_normalized'].describe())

In [ ]:
from scipy.spatial import cKDTree
import time

# Step 4: Create spatial index for snapping points to nearest road node

nodes = list(G.nodes())
node_coords = np.array(nodes)
tree = cKDTree(node_coords)

print(f"✅ Spatial index built for {len(nodes):,} road nodes")

# Test: how long does one shortest-path calculation take?
test_facility = (master.iloc[0]['Longitude'], master.iloc[0]['Latitude'])
_, idx = tree.query(test_facility)
facility_node = nodes[idx]

test_pop = (pop_df.iloc[0]['lon'], pop_df.iloc[0]['lat'])
_, idx2 = tree.query(test_pop)
pop_node = nodes[idx2]

start = time.time()
try:
    travel_time = nx.shortest_path_length(G, facility_node, pop_node, weight='weight')
    elapsed = time.time() - start
    print(f"\nTest shortest path: {round(travel_time, 1)} minutes")
    print(f"Calculation time: {round(elapsed, 3)} seconds")
except nx.NvetworkXNoPath:
    elapsed = time.time() - start
    print(f"\nNo path found ({round(elapsed, 3)} seconds)")

print(f"\nIf each calculation takes {round(elapsed, 3)}s:")
print(f"  9,978 facilities × 278,001 pop points = {9978 * 278001:,} calculations")
print(f"  That would take roughly {round(elapsed * 9978 * 278001 / 3600 / 24)} days")
print(f"\n  We need a smarter approach!")

In [ ]:
# Check network connectivity
components = list(nx.connected_components(G))
component_sizes = sorted([len(c) for c in components], reverse=True)

print(f"Total connected components: {len(components)}")
print(f"\nTop 10 component sizes:")
for i, size in enumerate(component_sizes[:10]):
    print(f"  Component {i+1}: {size:,} nodes")

print(f"\nMain network covers {round(component_sizes[0]/len(nodes)*100, 1)}% of all nodes")

That's a big problem. The network is completely fragmented — 209,921 disconnected pieces and the largest one only has 2.3% of all nodes. The road network isn't connected properly.
This happened because we only used the start and end points of each road segment. Many roads share intermediate points but we didn't capture those, so roads that should be connected aren't.
We need to rebuild the graph using ALL points along each road segment, not just the start and end:

In [ ]:
# Rebuild the graph using all intermediate points along each road

print("Rebuilding road network with intermediate points... (this will take a few minutes)")

G = nx.Graph()

for _, row in driveable.iterrows():
    coords = list(row.geometry.coords)
    speed = row['speed_kmh']
    
    # Connect every consecutive pair of points along the road
    for i in range(len(coords) - 1):
        start = coords[i]
        end = coords[i + 1]
        
        # Calculate distance between consecutive points
        dlat = (end[1] - start[1]) * 111320
        dlon = (end[0] - start[0]) * 111320 * np.cos(np.radians((start[1] + end[1]) / 2))
        distance_m = np.sqrt(dlat**2 + dlon**2)
        
        # Travel time in minutes
        travel_time = (distance_m / 1000) / speed * 60
        
        G.add_edge(start, end, weight=travel_time)

print(f"✅ Network rebuilt!")
print(f"   Nodes: {G.number_of_nodes():,}")
print(f"   Edges: {G.number_of_edges():,}")

# Check connectivity again
components = list(nx.connected_components(G))
component_sizes = sorted([len(c) for c in components], reverse=True)
print(f"\n   Connected components: {len(components)}")
print(f"   Largest component: {component_sizes[0]:,} nodes ({round(component_sizes[0]/G.number_of_nodes()*100, 1)}%)")

In [ ]:
# Rebuild spatial index with new nodes
nodes = list(G.nodes())
node_coords = np.array(nodes)
tree = cKDTree(node_coords)

# Keep only the main connected component for calculations
main_component = max(nx.connected_components(G), key=len)
print(f"Main component: {len(main_component):,} nodes")

# Test shortest path again
import time

test_facility = (master.iloc[0]['Longitude'], master.iloc[0]['Latitude'])
_, idx = tree.query(test_facility)
facility_node = nodes[idx]

test_pop = (pop_df.iloc[100]['lon'], pop_df.iloc[100]['lat'])
_, idx2 = tree.query(test_pop)
pop_node = nodes[idx2]

start = time.time()
try:
    travel_time = nx.shortest_path_length(G, facility_node, pop_node, weight='weight')
    elapsed = time.time() - start
    print(f"\nTest shortest path: {round(travel_time, 1)} minutes")
    print(f"Calculation time: {round(elapsed, 3)} seconds")
except nx.NetworkXNoPath:
    elapsed = time.time() - start
    print(f"\nNo path found ({round(elapsed, 3)} seconds)")

In [ ]:
# Test with cutoff — much faster because it stops early
start = time.time()

# Only find nodes within 60 minutes of the facility
nearby = nx.single_source_dijkstra_path_length(G, facility_node, cutoff=60, weight='weight')

elapsed = time.time() - start
print(f"Nodes reachable within 60 min: {len(nearby):,}")
print(f"Calculation time: {round(elapsed, 3)} seconds")
print(f"\nEstimated time for all 9,978 facilities:")
print(f"  {round(elapsed * 9978 / 3600, 1)} hours")

In [ ]:
pip install python-igraph

In [ ]:
import igraph as ig
import time

print("Building igraph network... (this may take a few minutes)")

# We need to map our coordinate nodes to integer IDs for igraph
node_to_id = {}
id_to_node = {}
edges = []
weights = []

for (start, end, data) in G.edges(data=True):
    if start not in node_to_id:
        node_to_id[start] = len(node_to_id)
        id_to_node[len(id_to_node)] = start
    if end not in node_to_id:
        node_to_id[end] = len(node_to_id)
        id_to_node[len(id_to_node)] = end
    
    edges.append((node_to_id[start], node_to_id[end]))
    weights.append(data['weight'])

g_ig = ig.Graph(n=len(node_to_id), edges=edges, directed=False)
g_ig.es['weight'] = weights

print(f"✅ igraph network built!")
print(f"   Nodes: {g_ig.vcount():,}")
print(f"   Edges: {g_ig.ecount():,}")

# Test speed — same facility as before
test_facility = (master.iloc[0]['Longitude'], master.iloc[0]['Latitude'])
_, idx = tree.query(test_facility)
facility_coord = nodes[idx]
facility_id = node_to_id[facility_coord]

start_time = time.time()
distances = g_ig.distances(source=facility_id, weights='weight')[0]

# Count nodes within 60 minutes
within_60 = sum(1 for d in distances if d <= 60)
elapsed = time.time() - start_time

print(f"\nTest: nodes within 60 min: {within_60:,}")
print(f"Calculation time: {round(elapsed, 3)} seconds")
print(f"\nEstimated time for all 9,978 facilities:")
print(f"  {round(elapsed * 9978 / 3600, 1)} hours")

In [ ]:
# Try neighborhood-based approach — only explore nodes within cutoff
start_time = time.time()

# Use shortest paths with cutoff by finding neighborhood first
result = g_ig.distances(source=facility_id, weights='weight')
elapsed = time.time() - start_time
print(f"Full distances: {round(elapsed, 3)} seconds")

# Alternative: use dijkstra directly with early stopping
# igraph doesn't have native cutoff, so let's try a different optimization
# Instead of running per-facility, batch facilities that are close together

# Actually, let's try the biggest optimization:
# Reduce the network by only keeping important roads for long distances

# First, how many facilities can we process in parallel?
# Let's test batching 10 facilities at once
start_time = time.time()
test_ids = []
for i in range(10):
    coord = (master.iloc[i]['Longitude'], master.iloc[i]['Latitude'])
    _, idx = tree.query(coord)
    test_ids.append(node_to_id[nodes[idx]])

results = g_ig.distances(source=test_ids, weights='weight')
elapsed = time.time() - start_time

print(f"\n10 facilities at once: {round(elapsed, 3)} seconds")
print(f"Per facility: {round(elapsed/10, 3)} seconds")
print(f"\nEstimated time for all 9,978 facilities:")
print(f"  {round(elapsed/10 * 9978 / 3600, 1)} hours")

In [ ]:
# Test bigger batch
start_time = time.time()

test_ids = []
for i in range(50):
    coord = (master.iloc[i]['Longitude'], master.iloc[i]['Latitude'])
    _, idx = tree.query(coord)
    test_ids.append(node_to_id[nodes[idx]])

results = g_ig.distances(source=test_ids, weights='weight')
elapsed = time.time() - start_time

print(f"50 facilities at once: {round(elapsed, 3)} seconds")
print(f"Per facility: {round(elapsed/50, 3)} seconds")
print(f"\nEstimated time for all 9,978 facilities:")
print(f"  {round(elapsed/50 * 9978 / 3600, 1)} hours")

In [ ]:
# Step 5: Snap all facilities to nearest road node

print("Snapping facilities to road network...")
facility_node_ids = []
for _, row in master.iterrows():
    coord = (row['Longitude'], row['Latitude'])
    _, idx = tree.query(coord)
    facility_node_ids.append(node_to_id[nodes[idx]])

master['node_id'] = facility_node_ids
print(f"✅ {len(facility_node_ids)} facilities snapped to road network")

# Step 6: Snap all population points to nearest road node
print("\nSnapping population points to road network... (this may take a minute)")
pop_node_ids = []
for _, row in pop_df.iterrows():
    coord = (row['lon'], row['lat'])
    _, idx = tree.query(coord)
    pop_node_ids.append(node_to_id[nodes[idx]])

pop_df['node_id'] = pop_node_ids
print(f"✅ {len(pop_node_ids)} population points snapped to road network")

In [ ]:
import pickle
import time

# E2SFCA Parameters
CUTOFF = 60  # minutes
BATCH_SIZE = 50

# Distance decay weights (Enhanced 2SFCA)
def decay_weight(travel_time):
    if travel_time <= 10:
        return 1.0
    elif travel_time <= 20:
        return 0.8
    elif travel_time <= 30:
        return 0.5
    elif travel_time <= 60:
        return 0.1
    else:
        return 0

# Build lookup: node_id -> list of (pop_df index, population)
print("Building population lookup...")
pop_node_lookup = {}
for i, row in pop_df.iterrows():
    nid = row['node_id']
    if nid not in pop_node_lookup:
        pop_node_lookup[nid] = []
    pop_node_lookup[nid].append((i, row['population_normalized']))

print(f"✅ Population lookup built")

# Initialize
facility_ratios = {}  # Step 1 results
accessibility_scores = np.zeros(len(pop_df))  # Step 2 results

total_batches = (len(master) + BATCH_SIZE - 1) // BATCH_SIZE
start_time = time.time()

print(f"\n=== RUNNING E2SFCA (both steps) ===")
print(f"Facilities: {len(master)}")
print(f"Population points: {len(pop_df)}")
print(f"Batches: {total_batches} (size {BATCH_SIZE})")
print(f"Estimated time: ~10-11 hours\n")

for batch_num in range(total_batches):
    start_idx = batch_num * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(master))
    
    batch_node_ids = master.iloc[start_idx:end_idx]['node_id'].tolist()
    
    # Calculate distances from all facilities in batch to all nodes
    results = g_ig.distances(source=batch_node_ids, weights='weight')
    
    for i, facility_idx in enumerate(range(start_idx, end_idx)):
        distances = results[i]
        
        # STEP 1: Calculate weighted population within catchment
        weighted_pop = 0
        catchment_pops = []  # store for step 2
        
        for nid in range(len(distances)):
            if distances[nid] <= CUTOFF:
                w = decay_weight(distances[nid])
                if w > 0 and nid in pop_node_lookup:
                    for pop_idx, pop in pop_node_lookup[nid]:
                        weighted_pop += pop * w
                        catchment_pops.append((pop_idx, w))
        
        # Facility ratio
        ratio = 1.0 / weighted_pop if weighted_pop > 0 else 0
        facility_ratios[facility_idx] = ratio
        
        # STEP 2: Add this facility's ratio to each population point in catchment
        for pop_idx, w in catchment_pops:
            accessibility_scores[pop_idx] += ratio * w
    
    # Progress update every 10 batches
    if (batch_num + 1) % 10 == 0 or batch_num == 0:
        elapsed = time.time() - start_time
        pct = (batch_num + 1) / total_batches
        eta_hours = (elapsed / pct - elapsed) / 3600
        print(f"  Batch {batch_num+1}/{total_batches} ({round(pct*100,1)}%) — ETA: {round(eta_hours, 1)} hours remaining")
    
    # Save progress every 50 batches
    if (batch_num + 1) % 50 == 0:
        save_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\e2sfca_progress.pkl"
        with open(save_path, 'wb') as f:
            pickle.dump({
                'facility_ratios': facility_ratios,
                'accessibility_scores': accessibility_scores,
                'last_batch': batch_num
            }, f)
        print(f"  💾 Progress saved")

# Final save
save_path = r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\e2sfca_complete.pkl"
with open(save_path, 'wb') as f:
    pickle.dump({
        'facility_ratios': facility_ratios,
        'accessibility_scores': accessibility_scores
    }, f)

total_time = (time.time() - start_time) / 3600
print(f"\n✅ E2SFCA COMPLETE!")
print(f"Total time: {round(total_time, 1)} hours")
print(f"Facilities processed: {len(facility_ratios)}")
print(f"Population points scored: {(accessibility_scores > 0).sum()}")

In [ ]:
# Add accessibility scores to population dataframe
pop_df['accessibility_score'] = accessibility_scores

# Basic stats
print("=== E2SFCA ACCESSIBILITY SCORES ===\n")
print(f"Population points with access (score > 0): {(pop_df['accessibility_score'] > 0).sum():,}")
print(f"Population points with NO access: {(pop_df['accessibility_score'] == 0).sum():,}")

scored = pop_df[pop_df['accessibility_score'] > 0]
print(f"\nAccessibility score stats (scored points only):")
print(scored['accessibility_score'].describe())

# How many people have no access?
no_access_pop = pop_df[pop_df['accessibility_score'] == 0]['population_normalized'].sum()
total_pop = pop_df['population_normalized'].sum()
print(f"\nPeople with NO access within 60 min: {int(no_access_pop):,} ({round(no_access_pop/total_pop*100, 1)}%)")

In [ ]:
# Save the accessibility scores
pop_df.to_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\e2sfca_accessibility_scores.csv", index=False)

print(f"✅ Saved: {len(pop_df)} population points with accessibility scores")
print(f"File: e2sfca_accessibility_scores.csv")

In [ ]:
# Let's see where the best and worst access is

# Top 10 best access grid cells
best = pop_df[pop_df['accessibility_score'] > 0].nlargest(10, 'accessibility_score')
print("=== BEST ACCESS (top 10 grid cells) ===\n")
print(f"{'Lat':<12} {'Lon':<12} {'Population':<15} {'Score'}")
print("="*50)
for _, row in best.iterrows():
    print(f"{row['lat']:<12.4f} {row['lon']:<12.4f} {int(row['population_normalized']):<15} {row['accessibility_score']:.6f}")

# Bottom 10 worst access (excluding zero)
worst = scored.nsmallest(10, 'accessibility_score')
print(f"\n=== WORST ACCESS (bottom 10 grid cells with some access) ===\n")
print(f"{'Lat':<12} {'Lon':<12} {'Population':<15} {'Score'}")
print("="*50)
for _, row in worst.iterrows():
    print(f"{row['lat']:<12.4f} {row['lon']:<12.4f} {int(row['population_normalized']):<15} {row['accessibility_score']:.6f}")

# The ratio between best and worst
best_score = pop_df['accessibility_score'].max()
worst_score = scored['accessibility_score'].min()
print(f"\nBest score: {best_score:.6f}")
print(f"Worst score: {worst_score:.6f}")
print(f"Ratio: {round(best_score/worst_score)}x difference")

In [ ]:
import geopandas as gpd

# Load district polygons
districts_gdf = gpd.read_file(r"C:\Users\hp\Downloads\gadm41_GHA_2.shp")

# Create GeoDataFrame from population points
pop_gdf = gpd.GeoDataFrame(
    pop_df,
    geometry=gpd.points_from_xy(pop_df['lon'], pop_df['lat']),
    crs="EPSG:4326"
)

# Spatial join — assign each population point to a district
districts_gdf = districts_gdf.to_crs("EPSG:4326")
pop_with_district = gpd.sjoin(pop_gdf, districts_gdf[['NAME_2', 'NAME_1', 'geometry']], how='left', predicate='within')

print(f"Population points mapped to districts: {pop_with_district['NAME_2'].notna().sum():,}")
print(f"Not in any district: {pop_with_district['NAME_2'].isna().sum():,}")

# Average accessibility score per district
district_access = pop_with_district.groupby(['NAME_2', 'NAME_1']).agg(
    Avg_Score=('accessibility_score', 'mean'),
    Population=('population_normalized', 'sum'),
    Grid_Cells=('accessibility_score', 'count'),
    No_Access_Cells=('accessibility_score', lambda x: (x == 0).sum())
).reset_index()

district_access.columns = ['District', 'Region', 'Avg_Score', 'Population', 'Grid_Cells', 'No_Access_Cells']
district_access = district_access.sort_values('Avg_Score', ascending=True)

print(f"\n=== TOP 10 WORST ACCESS DISTRICTS ===\n")
print(district_access.head(10).to_string())

print(f"\n=== TOP 10 BEST ACCESS DISTRICTS ===\n")
print(district_access.tail(10).to_string())

In [ ]:
# Save district accessibility scores
district_access.to_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\district_accessibility_scores.csv", index=False)

# Summary stats
print("=== NATIONAL SUMMARY ===\n")
total_no_access = pop_df[pop_df['accessibility_score'] == 0]['population_normalized'].sum()
total_pop = pop_df['population_normalized'].sum()
print(f"People with NO access within 60 min: {int(total_no_access):,} ({round(total_no_access/total_pop*100, 1)}%)")

# Districts with most no-access cells
print(f"\n=== DISTRICTS WITH MOST NO-ACCESS AREAS ===\n")
no_access_ranked = district_access.sort_values('No_Access_Cells', ascending=False).head(15)
print(f"{'District':<30} {'Region':<15} {'No Access Cells':<18} {'Total Cells':<15} {'% No Access'}")
print("="*90)
for _, row in no_access_ranked.iterrows():
    pct = round(row['No_Access_Cells'] / row['Grid_Cells'] * 100, 1)
    print(f"{row['District']:<30} {row['Region']:<15} {int(row['No_Access_Cells']):<18} {int(row['Grid_Cells']):<15} {pct}%")